# Cross-site scripting (XSS)

# Coding problem, 5.1.b: Fix the Reflected Comment

**Skill:** Mitigate Risk · **Maps to:** 5.1.A, 5.1.B · **Language:** Python 3 (pytest)

## The situation
`render_comment` builds the HTML that shows a visitor's comment on a page. It works for
ordinary text, but it drops the username and comment **straight into the HTML** by string
concatenation. That is reflected cross-site scripting (XSS): whatever the user typed is parsed
by the next visitor's browser as markup, so `<script>...</script>` in a comment becomes a live
script running in that visitor's session.

The starter is the vulnerable version. A comment of `<script>steal()</script>` comes back out
as a real `<script>` tag.

## Your task
Fix `render_comment` so user input can never be parsed as HTML. The correct defense is
**context-aware output encoding**: HTML-encode every user-supplied value before it goes into the
page, so `<` becomes `&lt;`, `>` becomes `&gt;`, `&` becomes `&amp;`, and the quote characters
are encoded too. Python's standard library does exactly this:

```python
import html
safe = html.escape(user_value, quote=True)
```

Encode the `username` and `comment` values, then place them into the same HTML template. Do
**not** detect, strip, or blocklist payloads — the point is that *encoded* input is harmless no
matter what it contains, while the app's own structural tags stay intact.

## Function contract
Implement `render_comment(username: str, comment: str) -> str`:
- Return the comment card HTML: `<div class="c"><b>USERNAME</b>: COMMENT</div>`, where
  `USERNAME` and `COMMENT` are the HTML-encoded user values.
- The structural tags (`<div class="c">`, `<b>`) are the app's own output and stay as real HTML.

A correct solution passes every test, including the injection cases. The concatenation version
fails them because the raw `<script>` and `<img onerror=...>` payloads reach the output as markup.

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def render_comment(username: str, comment: str) -> str
```

### Input
- `username`: the commenter's display name (untrusted input).
- `comment`: the comment body (untrusted input).

### Output
A single HTML string, the comment card:
```
<div class="c"><b>USERNAME</b>: COMMENT</div>
```
where `USERNAME` and `COMMENT` are the HTML-encoded forms of the two inputs.

### Encoding rules
- User values are HTML-encoded with `html.escape(value, quote=True)` semantics:
  `&` → `&amp;`, `<` → `&lt;`, `>` → `&gt;`, `"` → `&quot;`, `'` → `&#x27;`.
- The structural markup the function itself emits (`<div class="c">`, `<b>`, `</b>`, `</div>`)
  is real HTML and is **not** encoded.
- No detection, stripping, or blocklisting of input. Encoding alone is the defense.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only (`html`).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).
user values are encoded rather than concatenated raw.


In [ ]:
# Your implementation
import html  # noqa: F401  (you will need this to fix the function)


def render_comment(username: str, comment: str) -> str:
    """
    Return the HTML card that displays a visitor's comment:

        <div class="c"><b>USERNAME</b>: COMMENT</div>

    VULNERABLE starter: it concatenates the username and comment straight into
    the HTML, so a comment like <script>steal()</script> is sent to the next
    visitor's browser as a real script tag (reflected XSS).

    Fix it with context-aware OUTPUT ENCODING: HTML-encode each user value
    (html.escape(value, quote=True)) before placing it into the template, so the
    browser can never parse user input as markup. Do not strip or blocklist input;
    keep the structural <div>/<b> tags as real HTML.
    """
    return '<div class="c"><b>' + username + "</b>: " + comment + "</div>"


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

import html


# ---- ordinary content is shown normally, template stays intact -----------
def test_plain_comment_renders():
    out = impl.render_comment("jordan", "Nice, clear write-up")
    assert "jordan" in out
    assert "Nice, clear write-up" in out


def test_structural_tags_present():
    out = impl.render_comment("jordan", "hello")
    assert '<div class="c">' in out
    assert "<b>" in out and "</b>" in out and "</div>" in out


# ---- injection must be encoded, not reflected as markup (only encoding passes) ----
def test_script_payload_encoded():
    out = impl.render_comment("jordan", "<script>steal()</script>")
    assert "<script>" not in out
    assert "&lt;script&gt;" in out


def test_img_onerror_payload_encoded():
    out = impl.render_comment("jordan", '<img src=x onerror=steal()>')
    assert "<img" not in out
    assert "&lt;img" in out


def test_malicious_username_encoded():
    out = impl.render_comment("<script>evil()</script>", "hi")
    assert "<script>" not in out
    assert "&lt;script&gt;" in out


def test_quotes_and_ampersand_encoded():
    out = impl.render_comment("jordan", 'a "b" \'c\' & d')
    assert "&quot;" in out
    assert "&#x27;" in out
    assert "&amp;" in out


# ---- the user value is encoded exactly as html.escape(..., quote=True) ----
def test_user_value_encoded_exactly():
    payload = '"><svg/onload=steal()>&\''
    out = impl.render_comment("jordan", payload)
    assert html.escape(payload, quote=True) in out
    assert payload not in out


def test_ampersand_not_double_or_raw():
    out = impl.render_comment("a&b", "c")
    assert "&amp;" in out


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
